# Functional specialization smoke test

Quick end-to-end smoke test of retraining, ablation, and correlation specialization metrics on two RNN runs:
- `two_module_rnn_25_task_routed_sp07_nb2` (task-routed input)
- `two_module_rnn_25_sp07_nb2` (shared input)

For each run we:
1. Rebuild the trained wrapper from `settings.json` and `state_*.pt`.
2. Build A1-B-A2 loaders from participant trial data.
3. Train the probe readout for 1 epoch and compute retraining specialization.
4. Compute ablation specialization on the retrained probe.
5. Compute correlation specialization offline from `hiddens_per_module` in the `.npz` file.

## 1. Setup and imports

In [1]:
import os
import sys
from pathlib import Path
import numpy as np
import torch
from torch.utils.data import ConcatDataset, DataLoader

project_root = Path(os.getcwd()).resolve()
while not (project_root / "a1b2").exists():
    if project_root == project_root.parent:
        raise RuntimeError("Project root (containing a1b2) not found.")
    project_root = project_root.parent
sys.path.insert(0, str(project_root))

data_folder = project_root / "data"
sim_folder = data_folder / "simulations"

from a1b2.analysis import transfer_interference as ann
from a1b2.analysis import run_loader
from a1b2.analysis.retraining_a1b2 import (
    create_retraining_model_a1b2,
    train_probe_readout_a1b2,
    eval_probe_readout_a1b2,
    retraining_specialization_scalar,
    compute_ablations_metric_a1b2,
)
from a1b2.analysis.correlations_a1b2 import compute_correlation_metric_a1b2
from a1b2.data.basic_funcs import get_datasets
from a1b2.models.ffn import CreateParticipantDataset

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
project_root, data_folder, sim_folder, device

(PosixPath('/home/kat/workspace/Structure-Function-Analysis-of-Network-Topologies/a1b2_modular'),
 PosixPath('/home/kat/workspace/Structure-Function-Analysis-of-Network-Topologies/a1b2_modular/data'),
 PosixPath('/home/kat/workspace/Structure-Function-Analysis-of-Network-Topologies/a1b2_modular/data/simulations'),
 device(type='cuda'))

## 2. Resolve the two target runs

In [14]:
'''Resolve run folders whose names start with the given prefixes.
We only use the first match per prefix (print if multiple).'''
# for low sparse and no communication runs     
run_prefixes = [
    # No comms (25-dim nb2) — fix spelling: no_comms not no_comm
    "two_module_rnn_25_no_comms_nb2",
    "two_module_rnn_25_task_routed_no_comms_nb2",
    # Sp1 (25-dim nb2)
    "two_module_rnn_25_nb2",
    "two_module_rnn_25_task_routed_nb2",
]

def resolve_run(prefix):
    candidates = []
    if not sim_folder.exists():
        raise FileNotFoundError(f"Simulations folder not found: {sim_folder}")
    for p in sim_folder.iterdir():
        if p.is_dir() and p.name.startswith(prefix):
            candidates.append(p)
    if not candidates:
        raise ValueError(f"No run folder starting with {prefix!r} in {sim_folder}")
    candidates = sorted(candidates)
    if len(candidates) > 1:
        print(f"Multiple matches for {prefix!r}, using {candidates[0].name}; all: {[c.name for c in candidates]}")
    return candidates[0]

runs = {prefix: resolve_run(prefix) for prefix in run_prefixes}
runs

{'two_module_rnn_25_no_comms_nb2': PosixPath('/home/kat/workspace/Structure-Function-Analysis-of-Network-Topologies/a1b2_modular/data/simulations/two_module_rnn_25_no_comms_nb2_nb2_shared_sp0_sep_cr_RNN'),
 'two_module_rnn_25_task_routed_no_comms_nb2': PosixPath('/home/kat/workspace/Structure-Function-Analysis-of-Network-Topologies/a1b2_modular/data/simulations/two_module_rnn_25_task_routed_no_comms_nb2_nb2_task_routed_sp0_sep_cr_RNN'),
 'two_module_rnn_25_nb2': PosixPath('/home/kat/workspace/Structure-Function-Analysis-of-Network-Topologies/a1b2_modular/data/simulations/two_module_rnn_25_nb2_nb2_shared_sp1.0_sep_cr_RNN'),
 'two_module_rnn_25_task_routed_nb2': PosixPath('/home/kat/workspace/Structure-Function-Analysis-of-Network-Topologies/a1b2_modular/data/simulations/two_module_rnn_25_task_routed_nb2_nb2_task_routed_sp1_sep_cr_RNN')}

## 3. Helper: A1-B-A2 loader for one participant

In [15]:
def build_loader_for_participant(df, participant, task_parameters, batch_size=32, shuffle=False):
    """
    Build a single DataLoader over A1 + B + A2 trials for one participant,
    matching the training data format (input, label_x, label_y, feature_probe, etc.).
    """
    dataset_A1, dataset_B, dataset_A2, _, _ = get_datasets(df, participant, task_parameters)
    combined = ConcatDataset([
        CreateParticipantDataset(dataset_A1),
        CreateParticipantDataset(dataset_B),
        CreateParticipantDataset(dataset_A2),
    ])
    return DataLoader(combined, batch_size=batch_size, shuffle=shuffle)

## 4. Smoke test per run (retraining, ablations, correlation)

In [16]:
results = {}

df = ann.load_participant_data(str(data_folder))
participants_all = df["participant"].unique()

# #region agent log
_log_path = Path("/home/kat/workspace/Structure-Function-Analysis-of-Network-Topologies/.cursor/debug-8c816d.log")
def _dbg(msg, **data):
    import json as _j
    with open(_log_path, "a") as _f:
        _f.write(_j.dumps({"sessionId": "8c816d", "message": msg, "data": data, "timestamp": __import__("time").time_ns() // 1000000}) + "\n")
_dbg("participants_all from trial data", n=len(participants_all), types=[type(x).__name__ for x in (participants_all[:3].tolist() if len(participants_all) else [])], sample=participants_all[:5].tolist() if len(participants_all) else [])
# #endregion

for prefix, run_path in runs.items():
    print("=" * 80)
    print("Run:", run_path.name)
    sim_folder_run = run_path

    # Load settings and task parameters
    settings = run_loader.load_settings(sim_folder_run)
    task_parameters = settings.get("task_parameters") or ann.setup_task_parameters()

    # Participants: prefer both state+npz; if no state_*.pt, use npz-only for correlation-only
    participants_with_state = run_loader.list_participants_with_state(sim_folder_run)
    # #region agent log
    _sim_files = [n for n in __import__("os").listdir(sim_folder_run) if n.startswith("sim_") and n.endswith(".npz")]
    _state_files = [n for n in __import__("os").listdir(sim_folder_run) if n.startswith("state_") and n.endswith(".pt")]
    _dbg("run folder files", run=run_path.name, n_npz=len(_sim_files), n_state=len(_state_files), npz_sample=_sim_files[:3], state_sample=_state_files[:3])
    _dbg("list_participants_with_state", run=run_path.name, n=len(participants_with_state), sample=participants_with_state[:5] if participants_with_state else [])
    # #endregion
    participants = [p for p in participants_with_state if p in participants_all]
    correlation_only = False
    if not participants:
        participants_with_npz = run_loader.list_participants_with_npz(sim_folder_run)
        participants = [p for p in participants_with_npz if p in participants_all]
        correlation_only = bool(participants)
    # #region agent log
    _dbg("overlap", run=run_path.name, n_overlap=len(participants), sample=participants[:3] if participants else [], correlation_only=correlation_only)
    # #endregion
    if not participants:
        print("No overlapping participants (with state+npz or npz-only) and trial data; skipping run.")
        continue
    participant = participants[0]
    if correlation_only:
        print("Using participant (correlation-only; no state_*.pt in this run):", participant)
    else:
        print("Using participant:", participant)

    retr_scalar = None
    ab_scalar = None
    corr_scalar = None

    if not correlation_only:
        # Build loader
        loader = build_loader_for_participant(
            df, participant, task_parameters, batch_size=32, shuffle=True
        )

        # Rebuild trained wrapper and load weights
        wrapper = run_loader.build_wrapper_from_settings(settings, device=device)
        state_path = sim_folder_run / f"state_{participant}.pt"
        run_loader.load_wrapper_state(wrapper, state_path)

        # --- Retraining probe readout (1 epoch for speed) ---
        wrapper_retrain = create_retraining_model_a1b2(wrapper, device=device)
        wrapper_retrain = train_probe_readout_a1b2(
            wrapper_retrain, loader, settings.get("condition", {}),
            n_epochs=1, lr=1e-3, device=device,
        )
        acc = eval_probe_readout_a1b2(wrapper_retrain, loader, device=device)
        retr_scalar = float(
            retraining_specialization_scalar(acc[0, 0], acc[1, 0], acc[0, 1], acc[1, 1])
        )
        print("Retraining acc matrix (probes x features):")
        print(acc)
        print("Retraining specialization:", retr_scalar)

        # --- Ablations on retrained probe ---
        ab = compute_ablations_metric_a1b2(wrapper_retrain, loader, device=device)
        print("Ablation acc matrix (probes x features):")
        print(ab["acc"])
        ab_scalar = float(ab["ablation_specialization"])
        print("Ablation specialization:", ab_scalar)

    # --- Correlation specialization from npz ---
    npz_path = sim_folder_run / f"sim_{participant}.npz"
    if not npz_path.exists():
        print("No npz for participant; skipping correlation.")
    else:
        with np.load(npz_path, allow_pickle=True) as data:
            if "hiddens_per_module" in data and "probes" in data:
                participant_data = {
                    "hiddens_per_module": data["hiddens_per_module"],
                    "probes": data["probes"],
                    "inputs": data["inputs"],
                }
                corr_out = compute_correlation_metric_a1b2(participant_data, n_samples=5)
                corr_scalar = float(corr_out["correlation_specialization"])
                print("Correlation specialization:", corr_scalar)
            else:
                print("npz missing hiddens_per_module/probes; skipping correlation.")

    results[prefix] = {
        "participant": participant,
        "retraining_specialization": retr_scalar,
        "ablation_specialization": ab_scalar,
        "correlation_specialization": corr_scalar,
    }

results

Run: two_module_rnn_25_no_comms_nb2_nb2_shared_sp0_sep_cr_RNN
Using participant (correlation-only; no state_*.pt in this run): study1_far_sub1
Correlation specialization: 0.0
Run: two_module_rnn_25_task_routed_no_comms_nb2_nb2_task_routed_sp0_sep_cr_RNN
Using participant (correlation-only; no state_*.pt in this run): study1_far_sub1


/home/kat/workspace/Structure-Function-Analysis-of-Network-Topologies/.venv/lib/python3.10/site-packages/numpy/lib/function_base.py:2897: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/home/kat/workspace/Structure-Function-Analysis-of-Network-Topologies/.venv/lib/python3.10/site-packages/numpy/lib/function_base.py:2898: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


Correlation specialization: 0.5
Run: two_module_rnn_25_nb2_nb2_shared_sp1.0_sep_cr_RNN
Using participant (correlation-only; no state_*.pt in this run): study1_far_sub1
Correlation specialization: 0.0
Run: two_module_rnn_25_task_routed_nb2_nb2_task_routed_sp1_sep_cr_RNN
Using participant (correlation-only; no state_*.pt in this run): study1_far_sub1
Correlation specialization: 0.38754149511228786


{'two_module_rnn_25_no_comms_nb2': {'participant': 'study1_far_sub1',
  'retraining_specialization': None,
  'ablation_specialization': None,
  'correlation_specialization': 0.0},
 'two_module_rnn_25_task_routed_no_comms_nb2': {'participant': 'study1_far_sub1',
  'retraining_specialization': None,
  'ablation_specialization': None,
  'correlation_specialization': 0.5},
 'two_module_rnn_25_nb2': {'participant': 'study1_far_sub1',
  'retraining_specialization': None,
  'ablation_specialization': None,
  'correlation_specialization': 0.0},
 'two_module_rnn_25_task_routed_nb2': {'participant': 'study1_far_sub1',
  'retraining_specialization': None,
  'ablation_specialization': None,
  'correlation_specialization': 0.38754149511228786}}

In [ ]:
results = {}

df = ann.load_participant_data(str(data_folder))
participants_all = df["participant"].unique()

# #region agent log
_log_path = Path("/home/kat/workspace/Structure-Function-Analysis-of-Network-Topologies/.cursor/debug-8c816d.log")
def _dbg(msg, **data):
    import json as _j
    with open(_log_path, "a") as _f:
        _f.write(_j.dumps({"sessionId": "8c816d", "message": msg, "data": data, "timestamp": __import__("time").time_ns() // 1000000}) + "\n")
_dbg("participants_all from trial data", n=len(participants_all), types=[type(x).__name__ for x in (participants_all[:3].tolist() if len(participants_all) else [])], sample=participants_all[:5].tolist() if len(participants_all) else [])
# #endregion

for prefix, run_path in runs.items():
    print("=" * 80)
    print("Run:", run_path.name)
    sim_folder_run = run_path

    # Load settings and task parameters
    settings = run_loader.load_settings(sim_folder_run)
    task_parameters = settings.get("task_parameters") or ann.setup_task_parameters()

    # Participants: prefer both state+npz; if no state_*.pt, use npz-only for correlation-only
    participants_with_state = run_loader.list_participants_with_state(sim_folder_run)
    # #region agent log
    _sim_files = [n for n in __import__("os").listdir(sim_folder_run) if n.startswith("sim_") and n.endswith(".npz")]
    _state_files = [n for n in __import__("os").listdir(sim_folder_run) if n.startswith("state_") and n.endswith(".pt")]
    _dbg("run folder files", run=run_path.name, n_npz=len(_sim_files), n_state=len(_state_files), npz_sample=_sim_files[:3], state_sample=_state_files[:3])
    _dbg("list_participants_with_state", run=run_path.name, n=len(participants_with_state), sample=participants_with_state[:5] if participants_with_state else [])
    # #endregion
    participants = [p for p in participants_with_state if p in participants_all]
    correlation_only = False
    if not participants:
        participants_with_npz = run_loader.list_participants_with_npz(sim_folder_run)
        participants = [p for p in participants_with_npz if p in participants_all]
        correlation_only = bool(participants)
    # #region agent log
    _dbg("overlap", run=run_path.name, n_overlap=len(participants), sample=participants[:3] if participants else [], correlation_only=correlation_only)
    # #endregion
    if not participants:
        print("No overlapping participants (with state+npz or npz-only) and trial data; skipping run.")
        continue
    participant = participants[0]
    if correlation_only:
        print("Using participant (correlation-only; no state_*.pt in this run):", participant)
    else:
        print("Using participant:", participant)

    retr_scalar = None
    ab_scalar = None
    corr_scalar = None

    if not correlation_only:
        # Build loader
        loader = build_loader_for_participant(
            df, participant, task_parameters, batch_size=32, shuffle=True
        )

        # Rebuild trained wrapper and load weights
        wrapper = run_loader.build_wrapper_from_settings(settings, device=device)
        state_path = sim_folder_run / f"state_{participant}.pt"
        run_loader.load_wrapper_state(wrapper, state_path)

        # --- Retraining probe readout (1 epoch for speed) ---
        wrapper_retrain = create_retraining_model_a1b2(wrapper, device=device)
        wrapper_retrain = train_probe_readout_a1b2(
            wrapper_retrain, loader, settings.get("condition", {}),
            n_epochs=1, lr=1e-3, device=device,
        )
        acc = eval_probe_readout_a1b2(wrapper_retrain, loader, device=device)
        retr_scalar = float(
            retraining_specialization_scalar(acc[0, 0], acc[1, 0], acc[0, 1], acc[1, 1])
        )
        print("Retraining acc matrix (probes x features):")
        print(acc)
        print("Retraining specialization:", retr_scalar)

        # --- Ablations on retrained probe ---
        ab = compute_ablations_metric_a1b2(wrapper_retrain, loader, device=device)
        print("Ablation acc matrix (probes x features):")
        print(ab["acc"])
        ab_scalar = float(ab["ablation_specialization"])
        print("Ablation specialization:", ab_scalar)

    # --- Correlation specialization from npz ---
    npz_path = sim_folder_run / f"sim_{participant}.npz"
    if not npz_path.exists():
        print("No npz for participant; skipping correlation.")
    else:
        with np.load(npz_path, allow_pickle=True) as data:
            if "hiddens_per_module" in data and "probes" in data:
                participant_data = {
                    "hiddens_per_module": data["hiddens_per_module"],
                    "probes": data["probes"],
                    "inputs": data["inputs"],
                }
                corr_out = compute_correlation_metric_a1b2(participant_data, n_samples=5)
                corr_scalar = float(corr_out["correlation_specialization"])
                print("Correlation specialization:", corr_scalar)
            else:
                print("npz missing hiddens_per_module/probes; skipping correlation.")

    results[prefix] = {
        "participant": participant,
        "retraining_specialization": retr_scalar,
        "ablation_specialization": ab_scalar,
        "correlation_specialization": corr_scalar,
    }

results

Run: two_module_rnn_25_task_routed_low_sparse_nb2_nb2_task_routed_sp0.3_sep_cr_RNN
Using participant (correlation-only; no state_*.pt in this run): study1_far_sub1
Correlation specialization: 0.5902990075861723
Run: two_module_rnn_25_low_sparse_nb2_nb2_shared_sp0.3_sep_cr_RNN
Using participant (correlation-only; no state_*.pt in this run): study1_far_sub1
Correlation specialization: 0.0


{'two_module_rnn_25_task_routed_low_sparse_nb2': {'participant': 'study1_far_sub1',
  'retraining_specialization': None,
  'ablation_specialization': None,
  'correlation_specialization': 0.5902990075861723},
 'two_module_rnn_25_low_sparse_nb2': {'participant': 'study1_far_sub1',
  'retraining_specialization': None,
  'ablation_specialization': None,
  'correlation_specialization': 0.0}}

In [ ]:
results = {}

df = ann.load_participant_data(str(data_folder))
participants_all = df["participant"].unique()

# #region agent log
_log_path = Path("/home/kat/workspace/Structure-Function-Analysis-of-Network-Topologies/.cursor/debug-8c816d.log")
def _dbg(msg, **data):
    import json as _j
    with open(_log_path, "a") as _f:
        _f.write(_j.dumps({"sessionId": "8c816d", "message": msg, "data": data, "timestamp": __import__("time").time_ns() // 1000000}) + "\n")
_dbg("participants_all from trial data", n=len(participants_all), types=[type(x).__name__ for x in (participants_all[:3].tolist() if len(participants_all) else [])], sample=participants_all[:5].tolist() if len(participants_all) else [])
# #endregion

for prefix, run_path in runs.items():
    print("=" * 80)
    print("Run:", run_path.name)
    sim_folder_run = run_path

    # Load settings and task parameters
    settings = run_loader.load_settings(sim_folder_run)
    task_parameters = settings.get("task_parameters") or ann.setup_task_parameters()

    # Participants: prefer both state+npz; if no state_*.pt, use npz-only for correlation-only
    participants_with_state = run_loader.list_participants_with_state(sim_folder_run)
    # #region agent log
    _sim_files = [n for n in __import__("os").listdir(sim_folder_run) if n.startswith("sim_") and n.endswith(".npz")]
    _state_files = [n for n in __import__("os").listdir(sim_folder_run) if n.startswith("state_") and n.endswith(".pt")]
    _dbg("run folder files", run=run_path.name, n_npz=len(_sim_files), n_state=len(_state_files), npz_sample=_sim_files[:3], state_sample=_state_files[:3])
    _dbg("list_participants_with_state", run=run_path.name, n=len(participants_with_state), sample=participants_with_state[:5] if participants_with_state else [])
    # #endregion
    participants = [p for p in participants_with_state if p in participants_all]
    correlation_only = False
    if not participants:
        participants_with_npz = run_loader.list_participants_with_npz(sim_folder_run)
        participants = [p for p in participants_with_npz if p in participants_all]
        correlation_only = bool(participants)
    # #region agent log
    _dbg("overlap", run=run_path.name, n_overlap=len(participants), sample=participants[:3] if participants else [], correlation_only=correlation_only)
    # #endregion
    if not participants:
        print("No overlapping participants (with state+npz or npz-only) and trial data; skipping run.")
        continue
    participant = participants[0]
    if correlation_only:
        print("Using participant (correlation-only; no state_*.pt in this run):", participant)
    else:
        print("Using participant:", participant)

    retr_scalar = None
    ab_scalar = None
    corr_scalar = None

    if not correlation_only:
        # Build loader
        loader = build_loader_for_participant(
            df, participant, task_parameters, batch_size=32, shuffle=True
        )

        # Rebuild trained wrapper and load weights
        wrapper = run_loader.build_wrapper_from_settings(settings, device=device)
        state_path = sim_folder_run / f"state_{participant}.pt"
        run_loader.load_wrapper_state(wrapper, state_path)

        # --- Retraining probe readout (1 epoch for speed) ---
        wrapper_retrain = create_retraining_model_a1b2(wrapper, device=device)
        wrapper_retrain = train_probe_readout_a1b2(
            wrapper_retrain, loader, settings.get("condition", {}),
            n_epochs=1, lr=1e-3, device=device,
        )
        acc = eval_probe_readout_a1b2(wrapper_retrain, loader, device=device)
        retr_scalar = float(
            retraining_specialization_scalar(acc[0, 0], acc[1, 0], acc[0, 1], acc[1, 1])
        )
        print("Retraining acc matrix (probes x features):")
        print(acc)
        print("Retraining specialization:", retr_scalar)

        # --- Ablations on retrained probe ---
        ab = compute_ablations_metric_a1b2(wrapper_retrain, loader, device=device)
        print("Ablation acc matrix (probes x features):")
        print(ab["acc"])
        ab_scalar = float(ab["ablation_specialization"])
        print("Ablation specialization:", ab_scalar)

    # --- Correlation specialization from npz ---
    npz_path = sim_folder_run / f"sim_{participant}.npz"
    if not npz_path.exists():
        print("No npz for participant; skipping correlation.")
    else:
        with np.load(npz_path, allow_pickle=True) as data:
            if "hiddens_per_module" in data and "probes" in data:
                participant_data = {
                    "hiddens_per_module": data["hiddens_per_module"],
                    "probes": data["probes"],
                    "inputs": data["inputs"],
                }
                corr_out = compute_correlation_metric_a1b2(participant_data, n_samples=5)
                corr_scalar = float(corr_out["correlation_specialization"])
                print("Correlation specialization:", corr_scalar)
            else:
                print("npz missing hiddens_per_module/probes; skipping correlation.")

    results[prefix] = {
        "participant": participant,
        "retraining_specialization": retr_scalar,
        "ablation_specialization": ab_scalar,
        "correlation_specialization": corr_scalar,
    }

results

Run: two_module_rnn_25_task_routed_sp05_nb2_nb2_task_routed_sp0.5_sep_cr_RNN
Using participant (correlation-only; no state_*.pt in this run): study1_far_sub1
Correlation specialization: 0.5430412303481259
Run: two_module_rnn_25_sp05_nb2_nb2_shared_sp0.5_sep_cr_RNN
Using participant (correlation-only; no state_*.pt in this run): study1_far_sub1
Correlation specialization: 0.0


{'two_module_rnn_25_task_routed_sp05_nb2': {'participant': 'study1_far_sub1',
  'retraining_specialization': None,
  'ablation_specialization': None,
  'correlation_specialization': 0.5430412303481259},
 'two_module_rnn_25_sp05_nb2': {'participant': 'study1_far_sub1',
  'retraining_specialization': None,
  'ablation_specialization': None,
  'correlation_specialization': 0.0}}

In [ ]:
results = {}

df = ann.load_participant_data(str(data_folder))
participants_all = df["participant"].unique()

# #region agent log
_log_path = Path("/home/kat/workspace/Structure-Function-Analysis-of-Network-Topologies/.cursor/debug-8c816d.log")
def _dbg(msg, **data):
    import json as _j
    with open(_log_path, "a") as _f:
        _f.write(_j.dumps({"sessionId": "8c816d", "message": msg, "data": data, "timestamp": __import__("time").time_ns() // 1000000}) + "\n")
_dbg("participants_all from trial data", n=len(participants_all), types=[type(x).__name__ for x in (participants_all[:3].tolist() if len(participants_all) else [])], sample=participants_all[:5].tolist() if len(participants_all) else [])
# #endregion

for prefix, run_path in runs.items():
    print("=" * 80)
    print("Run:", run_path.name)
    sim_folder_run = run_path

    # Load settings and task parameters
    settings = run_loader.load_settings(sim_folder_run)
    task_parameters = settings.get("task_parameters") or ann.setup_task_parameters()

    # Participants: prefer both state+npz; if no state_*.pt, use npz-only for correlation-only
    participants_with_state = run_loader.list_participants_with_state(sim_folder_run)
    # #region agent log
    _sim_files = [n for n in __import__("os").listdir(sim_folder_run) if n.startswith("sim_") and n.endswith(".npz")]
    _state_files = [n for n in __import__("os").listdir(sim_folder_run) if n.startswith("state_") and n.endswith(".pt")]
    _dbg("run folder files", run=run_path.name, n_npz=len(_sim_files), n_state=len(_state_files), npz_sample=_sim_files[:3], state_sample=_state_files[:3])
    _dbg("list_participants_with_state", run=run_path.name, n=len(participants_with_state), sample=participants_with_state[:5] if participants_with_state else [])
    # #endregion
    participants = [p for p in participants_with_state if p in participants_all]
    correlation_only = False
    if not participants:
        participants_with_npz = run_loader.list_participants_with_npz(sim_folder_run)
        participants = [p for p in participants_with_npz if p in participants_all]
        correlation_only = bool(participants)
    # #region agent log
    _dbg("overlap", run=run_path.name, n_overlap=len(participants), sample=participants[:3] if participants else [], correlation_only=correlation_only)
    # #endregion
    if not participants:
        print("No overlapping participants (with state+npz or npz-only) and trial data; skipping run.")
        continue
    participant = participants[0]
    if correlation_only:
        print("Using participant (correlation-only; no state_*.pt in this run):", participant)
    else:
        print("Using participant:", participant)

    retr_scalar = None
    ab_scalar = None
    corr_scalar = None

    if not correlation_only:
        # Build loader
        loader = build_loader_for_participant(
            df, participant, task_parameters, batch_size=32, shuffle=True
        )

        # Rebuild trained wrapper and load weights
        wrapper = run_loader.build_wrapper_from_settings(settings, device=device)
        state_path = sim_folder_run / f"state_{participant}.pt"
        run_loader.load_wrapper_state(wrapper, state_path)

        # --- Retraining probe readout (1 epoch for speed) ---
        wrapper_retrain = create_retraining_model_a1b2(wrapper, device=device)
        wrapper_retrain = train_probe_readout_a1b2(
            wrapper_retrain, loader, settings.get("condition", {}),
            n_epochs=1, lr=1e-3, device=device,
        )
        acc = eval_probe_readout_a1b2(wrapper_retrain, loader, device=device)
        retr_scalar = float(
            retraining_specialization_scalar(acc[0, 0], acc[1, 0], acc[0, 1], acc[1, 1])
        )
        print("Retraining acc matrix (probes x features):")
        print(acc)
        print("Retraining specialization:", retr_scalar)

        # --- Ablations on retrained probe ---
        ab = compute_ablations_metric_a1b2(wrapper_retrain, loader, device=device)
        print("Ablation acc matrix (probes x features):")
        print(ab["acc"])
        ab_scalar = float(ab["ablation_specialization"])
        print("Ablation specialization:", ab_scalar)

    # --- Correlation specialization from npz ---
    npz_path = sim_folder_run / f"sim_{participant}.npz"
    if not npz_path.exists():
        print("No npz for participant; skipping correlation.")
    else:
        with np.load(npz_path, allow_pickle=True) as data:
            if "hiddens_per_module" in data and "probes" in data:
                participant_data = {
                    "hiddens_per_module": data["hiddens_per_module"],
                    "probes": data["probes"],
                    "inputs": data["inputs"],
                }
                corr_out = compute_correlation_metric_a1b2(participant_data, n_samples=5)
                corr_scalar = float(corr_out["correlation_specialization"])
                print("Correlation specialization:", corr_scalar)
            else:
                print("npz missing hiddens_per_module/probes; skipping correlation.")

    results[prefix] = {
        "participant": participant,
        "retraining_specialization": retr_scalar,
        "ablation_specialization": ab_scalar,
        "correlation_specialization": corr_scalar,
    }

results

Run: two_module_rnn_25_task_routed_sp07_nb2_nb2_task_routed_sp0.7_sep_cr_RNN
Using participant (correlation-only; no state_*.pt in this run): study1_far_sub1
Correlation specialization: 0.27264997443749944
Run: two_module_rnn_25_sp07_nb2_nb2_shared_sp0.7_sep_cr_RNN
Using participant (correlation-only; no state_*.pt in this run): study1_far_sub1
Correlation specialization: 0.0


{'two_module_rnn_25_task_routed_sp07_nb2': {'participant': 'study1_far_sub1',
  'retraining_specialization': None,
  'ablation_specialization': None,
  'correlation_specialization': 0.27264997443749944},
 'two_module_rnn_25_sp07_nb2': {'participant': 'study1_far_sub1',
  'retraining_specialization': None,
  'ablation_specialization': None,
  'correlation_specialization': 0.0}}